In [39]:
# ============================================================
# PROMETHEUS 2 JUDGE VALIDATION
# Imports and paths
# ============================================================

from pathlib import Path
import gc
import re
import json

import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/home/jovyan/project work/data_analyssis/fine tuning"
)

JUDGE_DIR = (
    PROJECT_DIR
    / "outputs"
    / "synthetic_generation"
    / "judge_validation"
)

REFERENCE_PATH = (
    JUDGE_DIR
    / "judge_validation_final_reference.csv"
)

INPUT_PATH = (
    JUDGE_DIR
    / "judge_validation_final_input.csv"
)

PROMETHEUS_OUTPUT_PATH = (
    JUDGE_DIR
    / "prometheus_validation_results.csv"
)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

PROMETHEUS_MODEL_ID = (
    "prometheus-eval/prometheus-7b-v2.0"
)


print("Reference file exists:", REFERENCE_PATH.exists())
print("Input file exists:", INPUT_PATH.exists())

print("\nPrometheus output will be saved to:")
print(PROMETHEUS_OUTPUT_PATH)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Reference file exists: True
Input file exists: True

Prometheus output will be saved to:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/judge_validation/prometheus_validation_results.csv

CUDA available: True
GPU: NVIDIA L40S


In [40]:
# ============================================================
# Load frozen judge-validation dataset
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_PATH
)

judge_input_df = pd.read_csv(
    INPUT_PATH
)


print("Reference dataset:", reference_df.shape)
print("Judge input dataset:", judge_input_df.shape)


print("\nReference columns:")
print(reference_df.columns.tolist())

print("\nJudge input columns:")
print(judge_input_df.columns.tolist())


display(judge_input_df.head())

Reference dataset: (80, 6)
Judge input dataset: (80, 4)

Reference columns:
['candidate_id', 'target_class_id', 'target_class', 'generated_text', 'human_decision', 'round']

Judge input columns:
['candidate_id', 'target_class_id', 'target_class', 'generated_text']


,candidate_id,target_class_id,target_class,generated_text
0,religious_high_precision_v1_0319,2,Religious Hate,o jb bnda bnda hua kis ko pta hi nahi
1,abusive_original_demo_v1_0100,0,Abusive/Offensive,saala lanti kuttay ki nasal
2,religious_high_precision_v1_0317,2,Religious Hate,agr apnay mullo k mujeed se pehly munafikat kr...
3,round2_religious_high_precision_v2_0616,2,Religious Hate,tum logon ne kaha tha k jb humne tera pichwaar...
4,round2_abusive_original_demo_v2_0916,0,Abusive/Offensive,yeh kya bhenchod


In [41]:
assert "human_decision" not in judge_input_df.columns

print(
    "Human labels hidden from Prometheus: OK"
)

Human labels hidden from Prometheus: OK


In [42]:
# ============================================================
# Original RUHSOLD class definitions
# ============================================================

RUHSOLD_DEFINITIONS = {

    "Abusive/Offensive": (
        "Profanity, strongly impolite, rude or vulgar language "
        "expressed with fighting or hurtful words in order to "
        "insult a targeted individual or group."
    ),

    "Sexism": (
        "Language used to express hatred towards a targeted "
        "individual or group based on gender or sexual orientation."
    ),

    "Religious Hate": (
        "Language used to express hatred towards a targeted "
        "individual or group based on their religious beliefs "
        "or lack of any religious beliefs, including the use "
        "of religion to incite violence or propagate hatred "
        "against a targeted individual or group."
    ),

    "Profane": (
        "The use of vulgar, foul or obscene language without "
        "an intended target."
    ),

    "Normal": (
        "Text that does not fall into the above categories."
    ),
}


for label, definition in RUHSOLD_DEFINITIONS.items():
    print(f"\n{label}:")
    print(definition)


Abusive/Offensive:
Profanity, strongly impolite, rude or vulgar language expressed with fighting or hurtful words in order to insult a targeted individual or group.

Sexism:
Language used to express hatred towards a targeted individual or group based on gender or sexual orientation.

Religious Hate:
Language used to express hatred towards a targeted individual or group based on their religious beliefs or lack of any religious beliefs, including the use of religion to incite violence or propagate hatred against a targeted individual or group.

Profane:
The use of vulgar, foul or obscene language without an intended target.

Normal:
Text that does not fall into the above categories.


In [43]:
# ============================================================
# Prometheus evaluation rubric
# ============================================================

JUDGE_PROMPT_VERSION = "prometheus_v1_ruhsold"

PROMETHEUS_SCORE_RUBRIC = """
Evaluation criterion:
Evaluate whether the response is suitable for inclusion as a synthetic
training example of the intended RUHSOLD class.

A suitable response should:
- be understandable Roman Urdu written primarily in the Latin alphabet;
- allow natural English code-mixing, informal spelling, abbreviations,
  and minor grammatical errors;
- be a usable social-media-style utterance rather than model commentary,
  instructions, corrupted output, or severely incomplete text;
- clearly satisfy the definition of the intended RUHSOLD class.

Class consistency is the most important requirement.

Do not reject a response merely because it is short, informal,
grammatically imperfect, or contains Roman Urdu spelling variation.

Do not assign a class merely because a class-associated word appears.
For example, mentioning a religion does not automatically constitute
Religious Hate, and mentioning a woman or using a gendered swear word
does not automatically constitute Sexism.

Score 1:
Clearly unsuitable. The response fundamentally fails the intended
RUHSOLD class, is seriously incoherent or corrupted, or is unusable
as a training example.

Score 2:
Mostly unsuitable. A major class-consistency or quality problem is
present. The response may contain words associated with the intended
class, but its underlying meaning does not genuinely satisfy that class.

Score 3:
Borderline or ambiguous. Some characteristics of the intended class
are present, but the correct class is not sufficiently clear, another
RUHSOLD class could reasonably fit better, or a noticeable generation
quality problem remains.

Score 4:
Suitable. The intended RUHSOLD class is clear and the response is usable
as a training example. Minor spelling errors, grammar errors,
code-mixing, informality, shortness, or slight awkwardness are acceptable.

Score 5:
Clearly suitable. The response is understandable, usable as an informal
Roman Urdu social-media example, and strongly and unambiguously satisfies
the intended RUHSOLD class without a substantial quality problem.
""".strip()


def prometheus_score_to_decision(score):

    if score >= 4:
        return "Accept"

    return "Reject"


print("Judge prompt version:", JUDGE_PROMPT_VERSION)

for score in range(1, 6):
    print(
        score,
        "->",
        prometheus_score_to_decision(score)
    )

Judge prompt version: prometheus_v1_ruhsold
1 -> Reject
2 -> Reject
3 -> Reject
4 -> Accept
5 -> Accept


In [44]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

print(
    "HF_HUB_DISABLE_XET =",
    os.environ["HF_HUB_DISABLE_XET"]
)

HF_HUB_DISABLE_XET = 1


In [45]:
import shutil
from pathlib import Path

home = Path.home()

total, used, free = shutil.disk_usage(home)

print(f"Total: {total / 1024**3:.2f} GB")
print(f"Used:  {used / 1024**3:.2f} GB")
print(f"Free:  {free / 1024**3:.2f} GB")

Total: 24.52 GB
Used:  21.94 GB
Free:  2.56 GB


In [39]:
!du -sh ~/.cache/huggingface 2>/dev/null
!du -sh ~/.cache/huggingface/hub 2>/dev/null
!du -sh ~/.cache/huggingface/xet 2>/dev/null

17G	/home/jovyan/.cache/huggingface
17G	/home/jovyan/.cache/huggingface/hub
796K	/home/jovyan/.cache/huggingface/xet


In [40]:
# ============================================================
# Inspect Hugging Face model cache
# ============================================================

!du -h --max-depth=1 ~/.cache/huggingface/hub | sort -hr

17G	/home/jovyan/.cache/huggingface/hub
14G	/home/jovyan/.cache/huggingface/hub/models--prometheus-eval--prometheus-7b-v2.0
1.2G	/home/jovyan/.cache/huggingface/hub/models--jhu-clsp--mmbert-base
1.1G	/home/jovyan/.cache/huggingface/hub/models--FacebookAI--xlm-roberta-base
685M	/home/jovyan/.cache/huggingface/hub/models--google-bert--bert-base-multilingual-cased
28K	/home/jovyan/.cache/huggingface/hub/.locks


In [41]:
# ============================================================
# Check Prometheus partial download
# ============================================================

!du -sh ~/.cache/huggingface/hub/models--prometheus-eval--prometheus-7b-v2.0 2>/dev/null

# Show incomplete files if present
!find ~/.cache/huggingface/hub/models--prometheus-eval--prometheus-7b-v2.0 \
    -type f -name "*.incomplete" -ls 2>/dev/null

14G	/home/jovyan/.cache/huggingface/hub/models--prometheus-eval--prometheus-7b-v2.0


In [12]:
# ============================================================
# Remove failed Prometheus partial download
# ============================================================

import shutil
from pathlib import Path

PROMETHEUS_CACHE = Path(
    "/home/jovyan/.cache/huggingface/hub/"
    "models--prometheus-eval--prometheus-7b-v2.0"
)

if PROMETHEUS_CACHE.exists():
    shutil.rmtree(PROMETHEUS_CACHE)
    print("Removed incomplete Prometheus cache.")
else:
    print("Prometheus cache not found.")

Removed incomplete Prometheus cache.


In [13]:
!df -h /home/jovyan
!du -sh ~/.cache/huggingface/hub

Filesystem      Size  Used Avail Use% Mounted on
/dev/rbd1        25G   22G  2.6G  90% /home/jovyan
17G	/home/jovyan/.cache/huggingface/hub


In [14]:
# ONLY run this if you are finished using Mistral for now

MISTRAL_CACHE = Path(
    "/home/jovyan/.cache/huggingface/hub/"
    "models--mistralai--Mistral-7B-Instruct-v0.2"
)

if MISTRAL_CACHE.exists():
    shutil.rmtree(MISTRAL_CACHE)
    print("Removed cached Mistral base model.")

Removed cached Mistral base model.


In [15]:
!df -h /home/jovyan
!du -h --max-depth=1 ~/.cache/huggingface/hub | sort -hr

Filesystem      Size  Used Avail Use% Mounted on
/dev/rbd1        25G  8.5G   17G  35% /home/jovyan
2.9G	/home/jovyan/.cache/huggingface/hub
1.2G	/home/jovyan/.cache/huggingface/hub/models--jhu-clsp--mmbert-base
1.1G	/home/jovyan/.cache/huggingface/hub/models--FacebookAI--xlm-roberta-base
685M	/home/jovyan/.cache/huggingface/hub/models--google-bert--bert-base-multilingual-cased
28K	/home/jovyan/.cache/huggingface/hub/.locks


In [16]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

print("HF_HUB_DISABLE_XET =", os.environ["HF_HUB_DISABLE_XET"])

HF_HUB_DISABLE_XET = 1


In [46]:
# ============================================================
# Load Prometheus 2 in 4-bit
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

PROMETHEUS_MODEL_ID = "prometheus-eval/prometheus-7b-v2.0"


# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

prometheus_tokenizer = AutoTokenizer.from_pretrained(
    PROMETHEUS_MODEL_ID,
    use_fast=True,
)

if prometheus_tokenizer.pad_token is None:
    prometheus_tokenizer.pad_token = (
        prometheus_tokenizer.eos_token
    )

prometheus_tokenizer.padding_side = "left"


# ------------------------------------------------------------
# 4-bit quantization
# ------------------------------------------------------------

prometheus_quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

prometheus_model = AutoModelForCausalLM.from_pretrained(
    PROMETHEUS_MODEL_ID,
    quantization_config=prometheus_quantization_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

prometheus_model.eval()

print("Prometheus loaded successfully.")
print("Device:", next(prometheus_model.parameters()).device)
print(
    "4-bit:",
    getattr(
        prometheus_model,
        "is_loaded_in_4bit",
        False
    )
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Prometheus loaded successfully.
Device: cuda:0
4-bit: True


In [47]:
# ============================================================
# Prometheus Direct Assessment system prompt
# ============================================================

PROMETHEUS_SYSTEM_PROMPT = """
You are a fair judge assistant tasked with providing clear,
objective feedback based on specific criteria, ensuring each
assessment reflects the absolute standards set for performance.
""".strip()

In [48]:
# ============================================================
# Format RUHSOLD definitions for each evaluation
# ============================================================

def format_ruhsold_definitions():

    return "\n".join(
        f"{label}: {definition}"
        for label, definition
        in RUHSOLD_DEFINITIONS.items()
    )

In [73]:
# ============================================================
# Build Prometheus direct-assessment prompt
# ============================================================

def build_prometheus_prompt(
    target_class,
    generated_text,
):
    """
    Build one reference-free Prometheus 2
    direct-assessment prompt.
    """

    definitions = format_ruhsold_definitions()

    orig_instruction = f"""
A synthetic Roman Urdu social-media post was generated for the
RUHSOLD hate-speech and offensive-language dataset.

Intended RUHSOLD class:
{target_class}

Evaluate whether the generated response is suitable for inclusion
as a training example of this intended class.

Use the following RUHSOLD class definitions:

{definitions}

The response should only be considered suitable when its meaning
genuinely matches the intended class definition.

Do not infer class consistency merely from isolated keywords.
For example, mentioning a religion does not automatically constitute
Religious Hate, and mentioning women or using a gendered swear word
does not automatically constitute Sexism.
""".strip()

    user_prompt = f"""
###Task Description:
An instruction (might include an Input inside it), a response to
evaluate, and a score rubric representing an evaluation criteria
are given.

1. Write a detailed feedback that assess the quality of the
response strictly based on the given score rubric, not evaluating
in general.

2. After writing a feedback, write a score that is an integer
between 1 and 5. You should refer to the score rubric.

3. The output format should look as follows:
"Feedback: (write a feedback for criteria)
[RESULT] (an integer number between 1 and 5)"

4. Please do not generate any other opening, closing, and explanations.

###The instruction to evaluate:
{orig_instruction}

###Response to evaluate:
{generated_text}

###Score Rubrics:
{PROMETHEUS_SCORE_RUBRIC}

###Feedback:
""".strip()

    return [
        {
            "role": "system",
            "content": PROMETHEUS_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

In [74]:
# ============================================================
# Inspect one Prometheus prompt
# ============================================================

sample = judge_input_df.iloc[0]

messages = build_prometheus_prompt(
    target_class=sample["target_class"],
    generated_text=sample["generated_text"],
)

print("TARGET CLASS:")
print(sample["target_class"])

print("\nGENERATED TEXT:")
print(sample["generated_text"])

print("\nSYSTEM PROMPT:")
print(messages[0]["content"])

print("\nUSER PROMPT:")
print(messages[1]["content"])

TARGET CLASS:
Religious Hate

GENERATED TEXT:
o jb bnda bnda hua kis ko pta hi nahi

SYSTEM PROMPT:
You are a fair judge assistant tasked with providing clear,
objective feedback based on specific criteria, ensuring each
assessment reflects the absolute standards set for performance.

USER PROMPT:
###Task Description:
An instruction (might include an Input inside it), a response to
evaluate, and a score rubric representing an evaluation criteria
are given.

1. Write a detailed feedback that assess the quality of the
response strictly based on the given score rubric, not evaluating
in general.

2. After writing a feedback, write a score that is an integer
between 1 and 5. You should refer to the score rubric.

3. The output format should look as follows:
"Feedback: (write a feedback for criteria)
[RESULT] (an integer number between 1 and 5)"

4. Please do not generate any other opening, closing, and explanations.

###The instruction to evaluate:
A synthetic Roman Urdu social-media post 

In [75]:
# ============================================================
# Generate one Prometheus judgment
# ============================================================

def generate_prometheus_judgment(
    target_class,
    generated_text,
    max_new_tokens=500,
):
    """
    Run Prometheus 2 direct assessment for one sample.
    Returns raw model output.
    """

    messages = build_prometheus_prompt(
        target_class=target_class,
        generated_text=generated_text,
    )

    prompt_text = prometheus_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = prometheus_tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(prometheus_model.device)

    with torch.no_grad():
        output_ids = prometheus_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=prometheus_tokenizer.eos_token_id,
        )

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]

    output_text = prometheus_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return output_text

In [76]:
# ============================================================
# Convert Prometheus score to binary decision
# ============================================================

def prometheus_score_to_decision(score):

    if score is None:
        return "Parse Error"

    if score >= 4:
        return "Accept"

    return "Reject"

In [77]:
# ============================================================
# Parse Prometheus score
# ============================================================

import re

def parse_prometheus_score(output_text):

    if output_text is None:
        return None

    # Preferred Prometheus format:
    # [RESULT] 4
    match = re.search(
        r"\[RESULT\]\s*:?\s*([1-5])",
        output_text,
        flags=re.IGNORECASE,
    )

    if match:
        return int(match.group(1))

    return None

In [78]:
# ============================================================
# Create small smoke-test set
# 4 Human Accept + 4 Human Reject
# ============================================================

SMOKE_TEST_SEED = 42

smoke_accept = (
    reference_df[
        reference_df["human_decision"] == "Accept"
    ]
    .sample(
        n=4,
        random_state=SMOKE_TEST_SEED
    )
)

smoke_reject = (
    reference_df[
        reference_df["human_decision"] == "Reject"
    ]
    .sample(
        n=4,
        random_state=SMOKE_TEST_SEED
    )
)

smoke_test_df = (
    pd.concat(
        [smoke_accept, smoke_reject],
        ignore_index=True
    )
    .sample(
        frac=1,
        random_state=SMOKE_TEST_SEED
    )
    .reset_index(drop=True)
)

print("Smoke-test samples:", len(smoke_test_df))

display(
    smoke_test_df[
        [
            "candidate_id",
            "target_class",
            "generated_text",
        ]
    ]
)

Smoke-test samples: 8


,candidate_id,target_class,generated_text
0,round2_abusive_original_demo_v2_0912,Abusive/Offensive,oye bhenchod
1,sexism_original_demo_v1_0531,Sexism,acha hai tu phir tere baare mei sharam nai hai
2,sexism_high_precision_v1_0518,Sexism,teri ammi ka jism meri marzi
3,profane_original_demo_v1_0716,Profane,psl huwa bc tera ghar mein
4,sexism_original_demo_v1_0538,Sexism,apnay baap ko sikhao to hijra
5,abusive_original_demo_v1_0103,Abusive/Offensive,chutiye ghatya ptm k charsi ho gaye
6,sexism_high_precision_v1_0508,Sexism,mard ko ghar nahi jana chahye jo usko aurat ke...
7,round2_profane_original_demo_v2_0814,Profane,rt : bc aik din kabhi tere saamne dekh raha th...


In [79]:
# ============================================================
# Run Prometheus on smoke-test samples
# ============================================================

smoke_results = []

for i, row in smoke_test_df.iterrows():

    print("=" * 80)
    print(f"Sample {i + 1}/8")
    print("=" * 80)

    raw_output = generate_prometheus_judgment(
        target_class=row["target_class"],
        generated_text=row["generated_text"],
    )

    score = parse_prometheus_score(
        raw_output
    )

    decision = prometheus_score_to_decision(
        score
    )

    smoke_results.append(
        {
            "candidate_id": row["candidate_id"],
            "target_class": row["target_class"],
            "generated_text": row["generated_text"],
            "prometheus_feedback": raw_output,
            "prometheus_score": score,
            "prometheus_decision": decision,
        }
    )

    print("Target class:", row["target_class"])
    print("Text:", row["generated_text"])
    print("\nPrometheus output:")
    print(raw_output)
    print("\nParsed score:", score)
    print("Prometheus decision:", decision)
    print()


smoke_results_df = pd.DataFrame(
    smoke_results
)

Sample 1/8
Target class: Abusive/Offensive
Text: oye bhenchod

Prometheus output:
The response provided is a single word, "oye bhenchod," which is a common Roman Urdu phrase. While it is a usable social-media-style utterance, it does not clearly satisfy the intended RUHSOLD class of Abusive/Offensive. The phrase is informal and could be interpreted as a friendly greeting or a term of endearment, rather than a clear expression of profanity or strongly impolite language. The phrase does not contain any explicit elements of hate speech, sexism, or religious hate, and it does not contain any profane language. Therefore, it does not meet the criteria for the intended class. However, it is important to note that the phrase could be used in a context where it is intended to be offensive, and in such a context, it could be considered suitable for inclusion in the Abusive/Offensive class. The response is understandable and usable as a social-media example, but it lacks the clear and unambiguous

In [80]:
# ============================================================
# Compare smoke-test predictions with human labels
# ============================================================

smoke_comparison = smoke_results_df.merge(
    reference_df[
        [
            "candidate_id",
            "human_decision",
        ]
    ],
    on="candidate_id",
    how="left",
)

smoke_comparison["agreement"] = (
    smoke_comparison[
        "prometheus_decision"
    ]
    ==
    smoke_comparison[
        "human_decision"
    ]
)

display(
    smoke_comparison[
        [
            "candidate_id",
            "target_class",
            "generated_text",
            "prometheus_score",
            "prometheus_decision",
            "human_decision",
            "agreement",
        ]
    ]
)

print(
    "Smoke-test agreement:",
    smoke_comparison["agreement"].mean()
)

,candidate_id,target_class,generated_text,prometheus_score,prometheus_decision,human_decision,agreement
0,round2_abusive_original_demo_v2_0912,Abusive/Offensive,oye bhenchod,2,Reject,Accept,False
1,sexism_original_demo_v1_0531,Sexism,acha hai tu phir tere baare mei sharam nai hai,4,Accept,Reject,False
2,sexism_high_precision_v1_0518,Sexism,teri ammi ka jism meri marzi,1,Reject,Accept,False
3,profane_original_demo_v1_0716,Profane,psl huwa bc tera ghar mein,3,Reject,Reject,True
4,sexism_original_demo_v1_0538,Sexism,apnay baap ko sikhao to hijra,1,Reject,Accept,False
5,abusive_original_demo_v1_0103,Abusive/Offensive,chutiye ghatya ptm k charsi ho gaye,1,Reject,Reject,True
6,sexism_high_precision_v1_0508,Sexism,mard ko ghar nahi jana chahye jo usko aurat ke...,5,Accept,Accept,True
7,round2_profane_original_demo_v2_0814,Profane,rt : bc aik din kabhi tere saamne dekh raha th...,3,Reject,Reject,True


Smoke-test agreement: 0.5


In [81]:
# ============================================================
# Inspect disagreements in the smoke test
# ============================================================

smoke_disagreements = smoke_comparison[
    smoke_comparison["agreement"] == False
].copy()

for _, row in smoke_disagreements.iterrows():

    print("=" * 100)

    print("Candidate ID:")
    print(row["candidate_id"])

    print("\nTarget class:")
    print(row["target_class"])

    print("\nGenerated text:")
    print(row["generated_text"])

    print("\nHuman decision:")
    print(row["human_decision"])

    print("\nPrometheus score:")
    print(row["prometheus_score"])

    print("\nPrometheus decision:")
    print(row["prometheus_decision"])

    print("\nPrometheus feedback:")
    print(row["prometheus_feedback"])

    print()

Candidate ID:
round2_abusive_original_demo_v2_0912

Target class:
Abusive/Offensive

Generated text:
oye bhenchod

Human decision:
Accept

Prometheus score:
2

Prometheus decision:
Reject

Prometheus feedback:
The response provided is a single word, "oye bhenchod," which is a common Roman Urdu phrase. While it is a usable social-media-style utterance, it does not clearly satisfy the intended RUHSOLD class of Abusive/Offensive. The phrase is informal and could be interpreted as a friendly greeting or a term of endearment, rather than a clear expression of profanity or strongly impolite language. The phrase does not contain any explicit elements of hate speech, sexism, or religious hate, and it does not contain any profane language. Therefore, it does not meet the criteria for the intended class. However, it is important to note that the phrase could be used in a context where it is intended to be offensive, and in such a context, it could be considered suitable for inclusion in the Abus

In [19]:
# ============================================================
# Run Prometheus evaluation on all 80 validation samples
# ============================================================

from tqdm.auto import tqdm
import pandas as pd

prometheus_results = []

for _, row in tqdm(
    judge_input_df.iterrows(),
    total=len(judge_input_df),
    desc="Prometheus evaluation"
):

    raw_output = generate_prometheus_judgment(
        target_class=row["target_class"],
        generated_text=row["generated_text"],
    )

    score = parse_prometheus_score(
        raw_output
    )

    decision = prometheus_score_to_decision(
        score
    )

    prometheus_results.append(
        {
            "candidate_id": row["candidate_id"],
            "target_class_id": row["target_class_id"],
            "target_class": row["target_class"],
            "generated_text": row["generated_text"],
            "prometheus_score": score,
            "prometheus_decision": decision,
            "prometheus_feedback": raw_output,
        }
    )


prometheus_results_df = pd.DataFrame(
    prometheus_results
)

print(
    "Completed evaluations:",
    len(prometheus_results_df)
)

print(
    "Parse errors:",
    prometheus_results_df[
        "prometheus_score"
    ].isna().sum()
)

display(
    prometheus_results_df.head()
)

Prometheus evaluation:   0%|          | 0/80 [00:00<?, ?it/s]

Completed evaluations: 80
Parse errors: 1


,candidate_id,target_class_id,target_class,generated_text,prometheus_score,prometheus_decision,prometheus_feedback
0,religious_high_precision_v1_0319,2,Religious Hate,o jb bnda bnda hua kis ko pta hi nahi,4.0,Accept,The response provided is a single sentence in ...
1,abusive_original_demo_v1_0100,0,Abusive/Offensive,saala lanti kuttay ki nasal,3.0,Reject,The response provided is a single sentence in ...
2,religious_high_precision_v1_0317,2,Religious Hate,agr apnay mullo k mujeed se pehly munafikat kr...,1.0,Reject,The response provided is a clear example of a ...
3,round2_religious_high_precision_v2_0616,2,Religious Hate,tum logon ne kaha tha k jb humne tera pichwaar...,NaN,Parse Error,The response provided is a clear example of a ...
4,round2_abusive_original_demo_v2_0916,0,Abusive/Offensive,yeh kya bhenchod,3.0,Reject,The response provided is a single sentence in ...


In [20]:
print(
    prometheus_results_df[
        "prometheus_score"
    ].value_counts(
        dropna=False
    ).sort_index()
)

print("\nPrometheus decisions:")

print(
    prometheus_results_df[
        "prometheus_decision"
    ].value_counts(
        dropna=False
    )
)

prometheus_score
1.0    28
2.0     3
3.0    22
4.0    13
5.0    13
NaN     1
Name: count, dtype: int64

Prometheus decisions:
prometheus_decision
Reject         53
Accept         26
Parse Error     1
Name: count, dtype: int64


In [21]:
parse_errors = prometheus_results_df[
    prometheus_results_df[
        "prometheus_score"
    ].isna()
]

print(
    "Number of parse errors:",
    len(parse_errors)
)

if len(parse_errors) > 0:

    for _, row in parse_errors.iterrows():

        print("=" * 100)
        print("Candidate ID:", row["candidate_id"])
        print("Target:", row["target_class"])
        print("Text:", row["generated_text"])

        print("\nRaw output:")
        print(row["prometheus_feedback"])

        print()

Number of parse errors: 1
Candidate ID: round2_religious_high_precision_v2_0616
Target: Religious Hate
Text: tum logon ne kaha tha k jb humne tera pichwaara ko dhoondha tha to tum logon ne goli maar diye..!

Raw output:
The response provided is a clear example of a text that is not suitable for inclusion as a training example of the intended RUHSOLD class. It is a fragment of a conversation that is not only incoherent but also lacks the necessary context to be considered a hate-speech or offensive-language example. The text does not contain any specific language or sentiment that would align with the definition of Religious Hate. It is a fragment of a conversation that is not only incoherent but also lacks the necessary context to be considered a hate-speech or offensive-language example. The response does not contain any specific language or sentiment that would align with the definition of Religious Hate. It is a fragment of a conversation that is not only incoherent but also lacks t

In [22]:
# ============================================================
# Separate valid Prometheus judgments from parse failures
# ============================================================

valid_prometheus_results = (
    prometheus_results_df[
        prometheus_results_df["prometheus_score"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

parse_failures = (
    prometheus_results_df[
        prometheus_results_df["prometheus_score"].isna()
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Valid Prometheus judgments:",
    len(valid_prometheus_results)
)

print(
    "Excluded parse failures:",
    len(parse_failures)
)

Valid Prometheus judgments: 79
Excluded parse failures: 1


In [23]:
valid_prometheus_results.to_csv(
    "prometheus_final_valid_results.csv",
    index=False,
    encoding="utf-8",
)

parse_failures.to_csv(
    "prometheus_parse_failures.csv",
    index=False,
    encoding="utf-8",
)

In [24]:
# ============================================================
# Merge Prometheus judgments with frozen human reference
# ============================================================

evaluation_df = valid_prometheus_results.merge(
    reference_df[
        [
            "candidate_id",
            "human_decision",
        ]
    ],
    on="candidate_id",
    how="inner",
    validate="one_to_one",
)

print("Valid evaluation samples:", len(evaluation_df))

print(
    "Missing human decisions:",
    evaluation_df["human_decision"].isna().sum()
)

print(
    "Missing Prometheus decisions:",
    evaluation_df["prometheus_decision"].isna().sum()
)

print("\nHuman distribution:")
print(
    evaluation_df["human_decision"].value_counts()
)

print("\nPrometheus distribution:")
print(
    evaluation_df["prometheus_decision"].value_counts()
)

Valid evaluation samples: 79
Missing human decisions: 0
Missing Prometheus decisions: 0

Human distribution:
human_decision
Accept    40
Reject    39
Name: count, dtype: int64

Prometheus distribution:
prometheus_decision
Reject    53
Accept    26
Name: count, dtype: int64


In [25]:
# ============================================================
# Overall Prometheus vs Human agreement
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

y_true = evaluation_df["human_decision"]
y_pred = evaluation_df["prometheus_decision"]

accuracy = accuracy_score(
    y_true,
    y_pred
)

accept_precision = precision_score(
    y_true,
    y_pred,
    pos_label="Accept",
    zero_division=0,
)

accept_recall = recall_score(
    y_true,
    y_pred,
    pos_label="Accept",
    zero_division=0,
)

accept_f1 = f1_score(
    y_true,
    y_pred,
    pos_label="Accept",
    zero_division=0,
)

kappa = cohen_kappa_score(
    y_true,
    y_pred
)

print("PROMETHEUS VS HUMAN")
print("=" * 45)

print(f"Samples evaluated:   {len(evaluation_df)}")
print(f"Coverage:            {len(evaluation_df)}/80 ({len(evaluation_df)/80:.2%})")

print(f"\nAccuracy:            {accuracy:.4f}")
print(f"Accept precision:    {accept_precision:.4f}")
print(f"Accept recall:       {accept_recall:.4f}")
print(f"Accept F1:           {accept_f1:.4f}")
print(f"Cohen's kappa:       {kappa:.4f}")

PROMETHEUS VS HUMAN
Samples evaluated:   79
Coverage:            79/80 (98.75%)

Accuracy:            0.5190
Accept precision:    0.5385
Accept recall:       0.3500
Accept F1:           0.4242
Cohen's kappa:       0.0421


In [26]:
# ============================================================
# Confusion matrix
# ============================================================

labels = [
    "Reject",
    "Accept",
]

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=labels,
)

confusion_df = pd.DataFrame(
    cm,
    index=[
        "Human Reject",
        "Human Accept",
    ],
    columns=[
        "Prometheus Reject",
        "Prometheus Accept",
    ],
)

print("Confusion matrix:")
display(confusion_df)

Confusion matrix:


,Prometheus Reject,Prometheus Accept
Human Reject,27,12
Human Accept,26,14


In [27]:
# ============================================================
# Class-wise Prometheus performance
# ============================================================

class_results = []

for target_class, group in evaluation_df.groupby(
    "target_class"
):

    y_true_class = group["human_decision"]
    y_pred_class = group["prometheus_decision"]

    class_accuracy = accuracy_score(
        y_true_class,
        y_pred_class
    )

    class_precision = precision_score(
        y_true_class,
        y_pred_class,
        pos_label="Accept",
        zero_division=0,
    )

    class_recall = recall_score(
        y_true_class,
        y_pred_class,
        pos_label="Accept",
        zero_division=0,
    )

    class_f1 = f1_score(
        y_true_class,
        y_pred_class,
        pos_label="Accept",
        zero_division=0,
    )

    class_kappa = cohen_kappa_score(
        y_true_class,
        y_pred_class
    )

    class_results.append(
        {
            "target_class": target_class,
            "n": len(group),
            "accuracy": class_accuracy,
            "accept_precision": class_precision,
            "accept_recall": class_recall,
            "accept_f1": class_f1,
            "cohen_kappa": class_kappa,
        }
    )

class_results_df = pd.DataFrame(
    class_results
)

display(
    class_results_df.round(4)
)

,target_class,n,accuracy,accept_precision,accept_recall,accept_f1,cohen_kappa
0,Abusive/Offensive,20,0.6500,1.0000,0.3,0.4615,0.3000
1,Profane,20,0.5000,0.5000,0.3,0.3750,0.0000
2,Religious Hate,19,0.5263,0.5455,0.6,0.5714,0.0447
3,Sexism,20,0.4000,0.3333,0.2,0.2500,-0.2000


In [28]:
from sklearn.metrics import classification_report

report = classification_report(
    evaluation_df["human_decision"],
    evaluation_df["prometheus_decision"],
    labels=["Accept", "Reject"],
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T

display(
    report_df[
        ["precision", "recall", "f1-score", "support"]
    ].round(4)
)

,precision,recall,f1-score,support
Accept,0.5385,0.3500,0.4242,40.000
Reject,0.5094,0.6923,0.5870,39.000
accuracy,0.5190,0.5190,0.5190,0.519
macro avg,0.5239,0.5212,0.5056,79.000
weighted avg,0.5241,0.5190,0.5046,79.000


In [32]:
from pathlib import Path
import os

print("Current working directory:")
print(Path.cwd())

print("\nFiles/folders in current directory:")
for item in sorted(Path.cwd().iterdir()):
    print(item)

Current working directory:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/judge_validation

Files/folders in current directory:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/judge_validation/.ipynb_checkpoints
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/judge_validation/Untitled.ipynb
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/judge_validation/judge_validation.ipynb
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/judge_validation/judge_validation_candidates.csv
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/judge_validation/judge_validation_candidates_round2.csv
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/judge_validation/judge_validation_final_input.csv
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/

In [33]:
from pathlib import Path

SEARCH_ROOT = Path(
    "/home/jovyan/project work/data_analyssis"
)

matches = []

for path in SEARCH_ROOT.rglob("RUHSOLD_validation*"):
    if path.is_file():
        matches.append(path)

print("Matching validation files:\n")

for path in matches:
    print(path)
    

Matching validation files:

/home/jovyan/project work/data_analyssis/RUHSOLD_validation.tsv
/home/jovyan/project work/data_analyssis/.ipynb_checkpoints/RUHSOLD_validation-checkpoint.tsv


In [37]:
# ============================================================
# PROMETHEUS EXPERIMENT 2
# Load authentic RUHSOLD validation set correctly
# ============================================================

from pathlib import Path
import pandas as pd

RUHSOLD_VALIDATION_PATH = Path(
    "/home/jovyan/project work/data_analyssis/RUHSOLD_validation.tsv"
)

validation_df = pd.read_csv(
    RUHSOLD_VALIDATION_PATH,
    sep="\t",
    header=None,
    names=["text", "label"]
)

print("Validation shape:", validation_df.shape)

print("\nColumns:")
print(validation_df.columns.tolist())

print("\nFirst 5 samples:")
display(validation_df.head())

print("\nLabel distribution:")
print(
    validation_df["label"]
    .value_counts()
    .sort_index()
)

Validation shape: (801, 2)

Columns:
['text', 'label']

First 5 samples:


,text,label
0,lahore ki chae or coffee se bi masa hai inko d...,3
1,rt : 15 minutes kya 30 minutes bhaunk sakta ha...,0
2,ustad ye wala scene yes kraoo,1
3,bc kya maha fuddu banda hai ye,0
4,'crazy foodie' 😂😂😂😂 itna sach,1



Label distribution:
label
0    192
1    428
2     63
3     67
4     51
Name: count, dtype: int64


In [38]:
# ============================================================
# RUHSOLD fine-grained label mapping
# ============================================================

ID_TO_LABEL = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}

LABEL_TO_ID = {
    label: label_id
    for label_id, label in ID_TO_LABEL.items()
}

validation_df["gold_label"] = (
    validation_df["label"]
    .astype(int)
    .map(ID_TO_LABEL)
)

print("Validation samples:", len(validation_df))

display(
    validation_df[
        ["text", "label", "gold_label"]
    ].head(10)
)

print("\nClass distribution:")
display(
    validation_df["gold_label"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

Validation samples: 801


,text,label,gold_label
0,lahore ki chae or coffee se bi masa hai inko d...,3,Sexism
1,rt : 15 minutes kya 30 minutes bhaunk sakta ha...,0,Abusive/Offensive
2,ustad ye wala scene yes kraoo,1,Normal
3,bc kya maha fuddu banda hai ye,0,Abusive/Offensive
4,'crazy foodie' 😂😂😂😂 itna sach,1,Normal
5,rt : abey yahoodi/cia/raw/mosad agent ganjay.,2,Religious Hate
6,zrori to ni ap mry sath thi tbhi wapsi ly aon...,1,Normal
7,man went from explaining bailly way better th...,4,Profane
8,akhrot (nuts) tor diye 😂😂. our girl knows whe...,0,Abusive/Offensive
9,50 gbs data bundle weekly nikalo 300 rupees m...,1,Normal



Class distribution:


,Class,Count
0,Normal,428
1,Abusive/Offensive,192
2,Sexism,67
3,Religious Hate,63
4,Profane,51


In [53]:
# ============================================================
# PROMETHEUS EXPERIMENT 2
# Create balanced authentic RUHSOLD diagnostic set
# ============================================================

DIAGNOSTIC_SEED = 42
SAMPLES_PER_CLASS = 20

prometheus_authentic_base = (
    validation_df
    .groupby(
        "gold_label",
        group_keys=False
    )
    .sample(
        n=SAMPLES_PER_CLASS,
        random_state=DIAGNOSTIC_SEED
    )
    .reset_index(drop=True)
)

print(
    "Authentic base samples:",
    len(prometheus_authentic_base)
)

print("\nClass distribution:")
display(
    prometheus_authentic_base[
        "gold_label"
    ]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

display(
    prometheus_authentic_base[
        [
            "text",
            "label",
            "gold_label",
        ]
    ].head(10)
)

Authentic base samples: 100

Class distribution:


,Class,Count
0,Abusive/Offensive,20
1,Normal,20
2,Profane,20
3,Religious Hate,20
4,Sexism,20


,text,label,gold_label
0,marro saali ko,0,Abusive/Offensive
1,sorry ap jaisi beghairt pr hum lanat bjhty hy,0,Abusive/Offensive
2,"ye hamari mumbai he kisike baap ki nahi, jo id...",0,Abusive/Offensive
3,apne baap ko bi blame kar bharwe to tujh jese ...,0,Abusive/Offensive
4,agar ap lahore se nh hain toh apka koi haq nh ...,0,Abusive/Offensive
5,bhenchod harami kutta ka bacha hain 😅 #mangoba...,0,Abusive/Offensive
6,abuse and cursing won’t cut it for me..... han...,0,Abusive/Offensive
7,teri aur tary harami shaikh rasheed ki ma ki c...,0,Abusive/Offensive
8,"kashmir hamara tha,hamara hain aur hamara hi r...",0,Abusive/Offensive
9,tu kyu beach me aaya re bosdk,0,Abusive/Offensive


In [54]:
# ============================================================
# Create positive + negative evaluator cases
# ============================================================

import random

ALL_RUHSOLD_CLASSES = [
    "Abusive/Offensive",
    "Normal",
    "Religious Hate",
    "Sexism",
    "Profane",
]

rng = random.Random(42)

diagnostic_cases = []

for sample_index, row in (
    prometheus_authentic_base.iterrows()
):

    text = row["text"]
    gold_class = row["gold_label"]

    # --------------------------------------------------------
    # Positive assignment
    # Correct RUHSOLD class
    # --------------------------------------------------------

    diagnostic_cases.append(
        {
            "source_index": sample_index,
            "text": text,
            "gold_class": gold_class,
            "proposed_class": gold_class,
            "assignment_type": "Positive",
            "expected_decision": "Accept",
        }
    )

    # --------------------------------------------------------
    # Negative assignment
    # Random incorrect RUHSOLD class
    # --------------------------------------------------------

    incorrect_classes = [
        label
        for label in ALL_RUHSOLD_CLASSES
        if label != gold_class
    ]

    wrong_class = rng.choice(
        incorrect_classes
    )

    diagnostic_cases.append(
        {
            "source_index": sample_index,
            "text": text,
            "gold_class": gold_class,
            "proposed_class": wrong_class,
            "assignment_type": "Negative",
            "expected_decision": "Reject",
        }
    )


prometheus_authentic_cases = pd.DataFrame(
    diagnostic_cases
)

print(
    "Total evaluator cases:",
    len(prometheus_authentic_cases)
)

print("\nExpected decisions:")
print(
    prometheus_authentic_cases[
        "expected_decision"
    ].value_counts()
)

print("\nAssignment types:")
print(
    prometheus_authentic_cases[
        "assignment_type"
    ].value_counts()
)

display(
    prometheus_authentic_cases.head(10)
)

Total evaluator cases: 200

Expected decisions:
expected_decision
Accept    100
Reject    100
Name: count, dtype: int64

Assignment types:
assignment_type
Positive    100
Negative    100
Name: count, dtype: int64


,source_index,text,gold_class,proposed_class,assignment_type,expected_decision
0,0,marro saali ko,Abusive/Offensive,Abusive/Offensive,Positive,Accept
1,0,marro saali ko,Abusive/Offensive,Normal,Negative,Reject
2,1,sorry ap jaisi beghairt pr hum lanat bjhty hy,Abusive/Offensive,Abusive/Offensive,Positive,Accept
3,1,sorry ap jaisi beghairt pr hum lanat bjhty hy,Abusive/Offensive,Normal,Negative,Reject
4,2,"ye hamari mumbai he kisike baap ki nahi, jo id...",Abusive/Offensive,Abusive/Offensive,Positive,Accept
5,2,"ye hamari mumbai he kisike baap ki nahi, jo id...",Abusive/Offensive,Sexism,Negative,Reject
6,3,apne baap ko bi blame kar bharwe to tujh jese ...,Abusive/Offensive,Abusive/Offensive,Positive,Accept
7,3,apne baap ko bi blame kar bharwe to tujh jese ...,Abusive/Offensive,Religious Hate,Negative,Reject
8,4,agar ap lahore se nh hain toh apka koi haq nh ...,Abusive/Offensive,Abusive/Offensive,Positive,Accept
9,4,agar ap lahore se nh hain toh apka koi haq nh ...,Abusive/Offensive,Religious Hate,Negative,Reject


In [55]:
# ============================================================
# Verify authentic diagnostic assignments
# ============================================================

negative_cases = prometheus_authentic_cases[
    prometheus_authentic_cases["assignment_type"] == "Negative"
].copy()

# Wrong class must never equal gold class
invalid_negative_cases = negative_cases[
    negative_cases["gold_class"] ==
    negative_cases["proposed_class"]
]

print(
    "Invalid negative assignments:",
    len(invalid_negative_cases)
)

print("\nDistribution of proposed wrong classes:")
display(
    negative_cases["proposed_class"]
    .value_counts()
    .rename_axis("Proposed wrong class")
    .reset_index(name="Count")
)

print("\nGold class × proposed wrong class:")
display(
    pd.crosstab(
        negative_cases["gold_class"],
        negative_cases["proposed_class"],
        margins=True
    )
)

Invalid negative assignments: 0

Distribution of proposed wrong classes:


,Proposed wrong class,Count
0,Normal,27
1,Religious Hate,21
2,Abusive/Offensive,19
3,Profane,18
4,Sexism,15



Gold class × proposed wrong class:


proposed_class,Abusive/Offensive,Normal,Profane,Religious Hate,Sexism,All
gold_class,,,,,,
Abusive/Offensive,0,8,3,7,2,20
Normal,7,0,4,3,6,20
Profane,5,5,0,8,2,20
Religious Hate,3,7,5,0,5,20
Sexism,4,7,6,3,0,20
All,19,27,18,21,15,100


In [60]:
# ============================================================
# PROMETHEUS EXPERIMENT 2
# Authentic RUHSOLD class-consistency evaluation prompt
# ============================================================

def build_prometheus_authentic_prompt(
    proposed_class,
    text,
):
    """
    Build a Prometheus direct-assessment prompt for evaluating
    whether an authentic RUHSOLD text is consistent with a
    proposed RUHSOLD class.
    """

    definitions = format_ruhsold_definitions()

    orig_instruction = f"""
An authentic Roman Urdu social-media post from the RUHSOLD dataset
is presented with a proposed RUHSOLD class.

Proposed RUHSOLD class:
{proposed_class}

Evaluate whether the text genuinely satisfies the definition of
the proposed class.

Use the following RUHSOLD class definitions:

{definitions}

Class consistency is the primary evaluation criterion.

Do not infer class consistency merely from isolated keywords.
For example, mentioning religion does not automatically constitute
Religious Hate, and mentioning women or using a gendered swear word
does not automatically constitute Sexism.

The text should receive a high score only when its meaning genuinely
matches the proposed RUHSOLD class.
""".strip()



    user_prompt = f"""
###Task Description:
An instruction (might include an Input inside it), a response to
evaluate, and a score rubric representing an evaluation criterion
are given.

1. Write detailed feedback that assesses the response strictly
   based on the given score rubric, not evaluating it in general.

2. After writing the feedback, write a score that is an integer
   between 1 and 5. You should refer to the score rubric.

3. The output format should look as follows:
   "Feedback: (write feedback for the criterion)
   [RESULT] (an integer number between 1 and 5)"

4. Please do not generate any other opening, closing, or explanation.

###The instruction to evaluate:
{orig_instruction}

###Response to evaluate:
{text}

###Score Rubrics:
{PROMETHEUS_AUTHENTIC_RUBRIC}

###Feedback:
""".strip()

    return [
        {
            "role": "system",
            "content": PROMETHEUS_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

In [61]:
# ============================================================
# Authentic RUHSOLD class-consistency rubric
# ============================================================

PROMETHEUS_AUTHENTIC_RUBRIC = """
Evaluation criterion:
Evaluate whether the response genuinely belongs to the proposed
RUHSOLD class according to the provided RUHSOLD class definitions.

Score 1:
Clearly inconsistent. The meaning of the response does not satisfy
the proposed RUHSOLD class and another class clearly fits better.

Score 2:
Mostly inconsistent. There may be a weak connection to the proposed
class, but the defining characteristics of that class are not
sufficiently present.

Score 3:
Borderline or ambiguous. Some characteristics of the proposed class
are present, but the class assignment is uncertain or another RUHSOLD
class could reasonably fit as well.

Score 4:
Consistent. The response clearly satisfies the definition of the
proposed RUHSOLD class, with only minor ambiguity.

Score 5:
Clearly consistent. The response strongly and unambiguously satisfies
the definition of the proposed RUHSOLD class.
""".strip()

In [62]:
# ============================================================
# Inspect one positive-negative pair
# ============================================================

sample_pair = prometheus_authentic_cases[
    prometheus_authentic_cases["source_index"] == 0
]

for _, row in sample_pair.iterrows():

    messages = build_prometheus_authentic_prompt(
        proposed_class=row["proposed_class"],
        text=row["text"],
    )

    print("=" * 100)
    print("TEXT:")
    print(row["text"])

    print("\nGOLD CLASS:")
    print(row["gold_class"])

    print("\nPROPOSED CLASS:")
    print(row["proposed_class"])

    print("\nASSIGNMENT:")
    print(row["assignment_type"])

    print("\nEXPECTED:")
    print(row["expected_decision"])

    print("\nPROMPT:")
    print(messages[1]["content"])

TEXT:
marro saali ko

GOLD CLASS:
Abusive/Offensive

PROPOSED CLASS:
Abusive/Offensive

ASSIGNMENT:
Positive

EXPECTED:
Accept

PROMPT:
###Task Description:
An instruction (might include an Input inside it), a response to
evaluate, and a score rubric representing an evaluation criterion
are given.

1. Write detailed feedback that assesses the response strictly
   based on the given score rubric, not evaluating it in general.

2. After writing the feedback, write a score that is an integer
   between 1 and 5. You should refer to the score rubric.

3. The output format should look as follows:
   "Feedback: (write feedback for the criterion)
   [RESULT] (an integer number between 1 and 5)"

4. Please do not generate any other opening, closing, or explanation.

###The instruction to evaluate:
An authentic Roman Urdu social-media post from the RUHSOLD dataset
is presented with a proposed RUHSOLD class.

Proposed RUHSOLD class:
Abusive/Offensive

Evaluate whether the text genuinely satisfies

In [63]:
# ============================================================
# Generate one authentic-data Prometheus judgment
# ============================================================

def generate_prometheus_authentic_judgment(
    proposed_class,
    text,
    max_new_tokens=500,
):

    messages = build_prometheus_authentic_prompt(
        proposed_class=proposed_class,
        text=text,
    )

    prompt_text = prometheus_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = prometheus_tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(prometheus_model.device)

    with torch.no_grad():

        output_ids = prometheus_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=prometheus_tokenizer.eos_token_id,
        )

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]

    output_text = prometheus_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return output_text

In [64]:
# ============================================================
# Authentic-data smoke test
# 3 texts × positive/negative = 6 cases
# ============================================================

smoke_source_indices = (
    prometheus_authentic_cases[
        "source_index"
    ]
    .drop_duplicates()
    .head(3)
)

authentic_smoke_df = (
    prometheus_authentic_cases[
        prometheus_authentic_cases[
            "source_index"
        ].isin(smoke_source_indices)
    ]
    .copy()
    .reset_index(drop=True)
)

authentic_smoke_results = []

for i, row in authentic_smoke_df.iterrows():

    raw_output = generate_prometheus_authentic_judgment(
        proposed_class=row["proposed_class"],
        text=row["text"],
    )

    score = parse_prometheus_score(
        raw_output
    )

    decision = prometheus_score_to_decision(
        score
    )

    authentic_smoke_results.append(
        {
            "source_index": row["source_index"],
            "text": row["text"],
            "gold_class": row["gold_class"],
            "proposed_class": row["proposed_class"],
            "assignment_type": row["assignment_type"],
            "expected_decision": row["expected_decision"],
            "prometheus_score": score,
            "prometheus_decision": decision,
            "prometheus_feedback": raw_output,
        }
    )

    print("=" * 100)
    print(f"Case {i + 1}/{len(authentic_smoke_df)}")
    print("Text:", row["text"])
    print("Gold:", row["gold_class"])
    print("Proposed:", row["proposed_class"])
    print("Expected:", row["expected_decision"])
    print("\nPrometheus:")
    print(raw_output)
    print("\nScore:", score)
    print("Decision:", decision)
    print()

Case 1/6
Text: marro saali ko
Gold: Abusive/Offensive
Proposed: Abusive/Offensive
Expected: Accept

Prometheus:
The response provided is a single sentence in Urdu, which is not sufficient to make a clear classification. The sentence is too short and lacks context, making it difficult to determine whether it is abusive, offensive, or normal. It does not contain any explicit language that would be considered profanity, strongly impolite, rude, or vulgar. Furthermore, it does not contain any language that could be interpreted as expressing hatred towards a targeted individual or group based on gender, sexual orientation, or religious beliefs. The response does not contain any language that could be considered profane or obscene. Therefore, it is not clear whether the response belongs to the proposed RUHSOLD class of Abusive/Offensive. The lack of context and the absence of any language that could be interpreted as abusive or offensive make the response ambiguous and not clearly consistent

In [82]:
# ============================================================
# PROMETHEUS EXPERIMENT 2
# Build 50 authentic evaluator cases
# 25 texts x (positive + negative) = 50 cases
# ============================================================

import random
import pandas as pd

AUTHENTIC_DIAGNOSTIC_SEED = 42
N_TEXTS = 25

# ------------------------------------------------------------
# Select 5 authentic samples from each RUHSOLD class
# ------------------------------------------------------------

authentic_25 = (
    validation_df
    .groupby(
        "gold_label",
        group_keys=False
    )
    .sample(
        n=5,
        random_state=AUTHENTIC_DIAGNOSTIC_SEED
    )
    .reset_index(drop=True)
)

print("Selected authentic texts:", len(authentic_25))

print("\nClass distribution:")
display(
    authentic_25["gold_label"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

# ------------------------------------------------------------
# Create positive + negative assignments
# ------------------------------------------------------------

ALL_RUHSOLD_CLASSES = [
    "Abusive/Offensive",
    "Normal",
    "Religious Hate",
    "Sexism",
    "Profane",
]

rng = random.Random(
    AUTHENTIC_DIAGNOSTIC_SEED
)

diagnostic_50_cases = []

for source_index, row in authentic_25.iterrows():

    text = row["text"]
    gold_class = row["gold_label"]

    # Positive case
    diagnostic_50_cases.append(
        {
            "source_index": source_index,
            "text": text,
            "gold_class": gold_class,
            "proposed_class": gold_class,
            "assignment_type": "Positive",
            "expected_decision": "Accept",
        }
    )

    # Negative case
    wrong_options = [
        label
        for label in ALL_RUHSOLD_CLASSES
        if label != gold_class
    ]

    wrong_class = rng.choice(
        wrong_options
    )

    diagnostic_50_cases.append(
        {
            "source_index": source_index,
            "text": text,
            "gold_class": gold_class,
            "proposed_class": wrong_class,
            "assignment_type": "Negative",
            "expected_decision": "Reject",
        }
    )


diagnostic_50_df = pd.DataFrame(
    diagnostic_50_cases
)

print("\nTotal evaluator cases:", len(diagnostic_50_df))

print("\nExpected decisions:")
print(
    diagnostic_50_df[
        "expected_decision"
    ].value_counts()
)

display(
    diagnostic_50_df.head(10)
)

Selected authentic texts: 25

Class distribution:


,Class,Count
0,Abusive/Offensive,5
1,Normal,5
2,Profane,5
3,Religious Hate,5
4,Sexism,5



Total evaluator cases: 50

Expected decisions:
expected_decision
Accept    25
Reject    25
Name: count, dtype: int64


,source_index,text,gold_class,proposed_class,assignment_type,expected_decision
0,0,marro saali ko,Abusive/Offensive,Abusive/Offensive,Positive,Accept
1,0,marro saali ko,Abusive/Offensive,Normal,Negative,Reject
2,1,sorry ap jaisi beghairt pr hum lanat bjhty hy,Abusive/Offensive,Abusive/Offensive,Positive,Accept
3,1,sorry ap jaisi beghairt pr hum lanat bjhty hy,Abusive/Offensive,Normal,Negative,Reject
4,2,"ye hamari mumbai he kisike baap ki nahi, jo id...",Abusive/Offensive,Abusive/Offensive,Positive,Accept
5,2,"ye hamari mumbai he kisike baap ki nahi, jo id...",Abusive/Offensive,Sexism,Negative,Reject
6,3,apne baap ko bi blame kar bharwe to tujh jese ...,Abusive/Offensive,Abusive/Offensive,Positive,Accept
7,3,apne baap ko bi blame kar bharwe to tujh jese ...,Abusive/Offensive,Religious Hate,Negative,Reject
8,4,agar ap lahore se nh hain toh apka koi haq nh ...,Abusive/Offensive,Abusive/Offensive,Positive,Accept
9,4,agar ap lahore se nh hain toh apka koi haq nh ...,Abusive/Offensive,Religious Hate,Negative,Reject


In [83]:
# ============================================================
# Run Prometheus on 50 authentic diagnostic cases
# ============================================================

from tqdm.auto import tqdm

authentic_50_results = []

for _, row in tqdm(
    diagnostic_50_df.iterrows(),
    total=len(diagnostic_50_df),
    desc="Prometheus authentic diagnostic"
):

    raw_output = generate_prometheus_authentic_judgment(
        proposed_class=row["proposed_class"],
        text=row["text"],
        max_new_tokens=500,
    )

    score = parse_prometheus_score(
        raw_output
    )

    decision = prometheus_score_to_decision(
        score
    )

    authentic_50_results.append(
        {
            "source_index": row["source_index"],
            "text": row["text"],
            "gold_class": row["gold_class"],
            "proposed_class": row["proposed_class"],
            "assignment_type": row["assignment_type"],
            "expected_decision": row["expected_decision"],
            "prometheus_score": score,
            "prometheus_decision": decision,
            "prometheus_feedback": raw_output,
        }
    )


authentic_50_results_df = pd.DataFrame(
    authentic_50_results
)

print(
    "Completed evaluations:",
    len(authentic_50_results_df)
)

print(
    "Parse errors:",
    authentic_50_results_df[
        "prometheus_score"
    ].isna().sum()
)

print("\nPrometheus decisions:")
print(
    authentic_50_results_df[
        "prometheus_decision"
    ].value_counts(
        dropna=False
    )
)

Prometheus authentic diagnostic:   0%|          | 0/50 [00:00<?, ?it/s]

Completed evaluations: 50
Parse errors: 0

Prometheus decisions:
prometheus_decision
Reject    38
Accept    12
Name: count, dtype: int64


In [84]:
# ============================================================
# Final evaluation:
# Prometheus vs expected authentic RUHSOLD assignments
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

eval_df = authentic_50_results_df.copy()

# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

y_true = eval_df["expected_decision"]
y_pred = eval_df["prometheus_decision"]

accuracy = accuracy_score(y_true, y_pred)

accept_precision = precision_score(
    y_true,
    y_pred,
    pos_label="Accept",
    zero_division=0,
)

accept_recall = recall_score(
    y_true,
    y_pred,
    pos_label="Accept",
    zero_division=0,
)

accept_f1 = f1_score(
    y_true,
    y_pred,
    pos_label="Accept",
    zero_division=0,
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0,
)

kappa = cohen_kappa_score(
    y_true,
    y_pred
)

print("=" * 60)
print("PROMETHEUS AUTHENTIC RUHSOLD DIAGNOSTIC")
print("=" * 60)

print(f"Samples evaluated:   {len(eval_df)}")
print(f"Accuracy:            {accuracy:.4f}")
print(f"Accept precision:    {accept_precision:.4f}")
print(f"Accept recall:       {accept_recall:.4f}")
print(f"Accept F1:           {accept_f1:.4f}")
print(f"Macro F1:            {macro_f1:.4f}")
print(f"Cohen's kappa:       {kappa:.4f}")

PROMETHEUS AUTHENTIC RUHSOLD DIAGNOSTIC
Samples evaluated:   50
Accuracy:            0.5800
Accept precision:    0.6667
Accept recall:       0.3200
Accept F1:           0.4324
Macro F1:            0.5495
Cohen's kappa:       0.1600


In [85]:
# ============================================================
# Confusion matrix
# ============================================================

labels = ["Reject", "Accept"]

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Expected Reject",
        "Expected Accept"
    ],
    columns=[
        "Prometheus Reject",
        "Prometheus Accept"
    ]
)

print("\nConfusion matrix:")
display(cm_df)


Confusion matrix:


,Prometheus Reject,Prometheus Accept
Expected Reject,21,4
Expected Accept,17,8


In [86]:
# ============================================================
# Results by original RUHSOLD gold class
# ============================================================

class_results = []

for gold_class, group in eval_df.groupby(
    "gold_class"
):

    class_accuracy = accuracy_score(
        group["expected_decision"],
        group["prometheus_decision"]
    )

    class_accept_recall = recall_score(
        group["expected_decision"],
        group["prometheus_decision"],
        pos_label="Accept",
        zero_division=0,
    )

    class_reject_recall = recall_score(
        group["expected_decision"],
        group["prometheus_decision"],
        pos_label="Reject",
        zero_division=0,
    )

    class_results.append(
        {
            "RUHSOLD class": gold_class,
            "Cases": len(group),
            "Accuracy": class_accuracy,
            "Correct-class acceptance rate": class_accept_recall,
            "Wrong-class rejection rate": class_reject_recall,
        }
    )

class_results_df = pd.DataFrame(
    class_results
)

display(
    class_results_df.round(4)
)

,RUHSOLD class,Cases,Accuracy,Correct-class acceptance rate,Wrong-class rejection rate
0,Abusive/Offensive,10,0.5,0.2,0.8
1,Normal,10,0.9,0.8,1.0
2,Profane,10,0.2,0.0,0.4
3,Religious Hate,10,0.7,0.4,1.0
4,Sexism,10,0.6,0.2,1.0


In [87]:
# ============================================================
# Save Prometheus authentic diagnostic results
# ============================================================

authentic_50_results_df.to_csv(
    "prometheus_authentic_ruhsold_diagnostic.csv",
    index=False
)

class_results_df.to_csv(
    "prometheus_authentic_ruhsold_class_results.csv",
    index=False
)

print("Prometheus diagnostic results saved.")

Prometheus diagnostic results saved.


In [89]:
""" Prometheus Authentic-RUHSOLD Diagnostic

Prometheus-7B was additionally evaluated on 25 authentic RUHSOLD
validation examples, balanced across the five fine-grained classes.
Each text was evaluated twice: once with its correct gold class
(positive assignment) and once with a randomly selected incorrect
class (negative assignment), producing 50 balanced evaluator cases.

Prometheus achieved 58.0% accuracy, 54.95% macro-F1, an Accept-class
F1 of 43.24%, and Cohen's kappa of 0.160. The evaluator correctly
rejected 21/25 (84%) incorrect class assignments, but accepted only
8/25 (32%) correct assignments, indicating a strong tendency toward
rejection.

Correct-class acceptance was particularly low for the harmful
fine-grained categories: Abusive/Offensive (20%), Profane (0%),
Religious Hate (40%), and Sexism (20%), compared with Normal (80%).

These results, together with the low agreement observed against
human judgments on synthetic samples, indicate that Prometheus-7B
is not sufficiently reliable as a class-consistency filter for
fine-grained Roman Urdu synthetic-data quality control. It was
therefore retained as a diagnostic evaluator experiment rather than
used as a filtering stage in the final augmentation pipeline."""

" Prometheus Authentic-RUHSOLD Diagnostic\n\nPrometheus-7B was additionally evaluated on 25 authentic RUHSOLD\nvalidation examples, balanced across the five fine-grained classes.\nEach text was evaluated twice: once with its correct gold class\n(positive assignment) and once with a randomly selected incorrect\nclass (negative assignment), producing 50 balanced evaluator cases.\n\nPrometheus achieved 58.0% accuracy, 54.95% macro-F1, an Accept-class\nF1 of 43.24%, and Cohen's kappa of 0.160. The evaluator correctly\nrejected 21/25 (84%) incorrect class assignments, but accepted only\n8/25 (32%) correct assignments, indicating a strong tendency toward\nrejection.\n\nCorrect-class acceptance was particularly low for the harmful\nfine-grained categories: Abusive/Offensive (20%), Profane (0%),\nReligious Hate (40%), and Sexism (20%), compared with Normal (80%).\n\nThese results, together with the low agreement observed against\nhuman judgments on synthetic samples, indicate that Prometheus-7